# OdontoHistoria IA - Modelo de Machine Learning (PEA 4)
Clasificador de fotos intraorales: **con caries / sin caries**. Clínica Fassiara.

Dataset: *Annotated intraoral image dataset for dental caries detection* (Zenodo, CC BY 4.0).

**Ejecuta las celdas una por una, de arriba hacia abajo.** Es una herramienta de apoyo: el diagnóstico lo hace el odontólogo.

## Celda 1 - Entorno (operación 13)

In [ ]:
import os, sys, platform, random, json
import numpy as np
import tensorflow as tf
print('Python:', sys.version.split()[0])
print('Sistema:', platform.system())
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## Celda 2 - Descargar el dataset (unos 1.6 GB, tarda unos minutos)

In [ ]:
!wget -q -c -O data_tree.txt "https://zenodo.org/records/14827784/files/Data%20tree.txt?download=1"
!head -n 60 data_tree.txt
!wget -q -c --show-progress -O Dataset.zip "https://zenodo.org/records/14827784/files/Dataset.zip?download=1"
!ls -lh Dataset.zip

In [ ]:
!unzip -q -o Dataset.zip -d /content/data
print('Descomprimido en /content/data')

## Celda 3 - Conocer el dataset

In [ ]:
from collections import Counter
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
imagenes, etiquetas = [], {}
for raiz, _, archivos in os.walk('/content/data'):
    for a in archivos:
        ruta = os.path.join(raiz, a)
        stem, ext = os.path.splitext(a)
        ext = ext.lower()
        if ext in IMG_EXT:
            imagenes.append(ruta)
        elif ext == '.txt':
            etiquetas.setdefault(stem, ruta)
print('Imagenes encontradas  :', len(imagenes))
print('Archivos .txt (YOLO)  :', len(etiquetas))
print('Extensiones           :', Counter(os.path.splitext(i)[1].lower() for i in imagenes))
print('Carpetas de primer nivel:', sorted(os.listdir('/content/data')))
if len(etiquetas) == 0:
    print('AVISO: no se encontraron etiquetas .txt. Copia esta salida y mandamela.')

## Celda 4 - Armar las etiquetas (con caries = 1, sin caries = 0)
Una imagen es **con caries** si su archivo de etiquetas existe y tiene al menos un recuadro.

In [ ]:
import pandas as pd
filas = []
for ruta in imagenes:
    stem = os.path.splitext(os.path.basename(ruta))[0]
    lab = etiquetas.get(stem)
    tiene = 0
    if lab and os.path.getsize(lab) > 0:
        with open(lab) as f:
            tiene = 1 if any(l.strip() for l in f) else 0
    filas.append({'ruta': ruta, 'caries': tiene})
df = pd.DataFrame(filas)
print(df['caries'].value_counts().rename({1: 'con caries', 0: 'sin caries'}))
if df['caries'].nunique() < 2:
    raise SystemExit('Solo hay una clase en los datos. Mandame la salida de las celdas 3 y 4 para ajustar el modelo.')

## Celda 5 - Dividir en entrenamiento, validación y prueba (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split
train_df, resto = train_test_split(df, test_size=0.30, stratify=df['caries'], random_state=SEED)
val_df, test_df = train_test_split(resto, test_size=0.50, stratify=resto['caries'], random_state=SEED)
for nombre, d in [('entrenamiento', train_df), ('validacion', val_df), ('prueba', test_df)]:
    print(f'{nombre:14s}: {len(d):5d} imagenes | con caries: {d.caries.sum()}')

## Celda 6 - Preparar los datos

In [ ]:
IMG = 224
BATCH = 32
def cargar(ruta, y):
    img = tf.io.read_file(ruta)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))
    return img, tf.cast(y, tf.float32)

def dataset(d, mezclar=False):
    ds = tf.data.Dataset.from_tensor_slices((d['ruta'].values, d['caries'].values))
    if mezclar:
        ds = ds.shuffle(len(d), seed=SEED)
    return ds.map(cargar, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_train, ds_val, ds_test = dataset(train_df, True), dataset(val_df), dataset(test_df)
print('Listo')

## Celda 7 - Crear el modelo (operación 14): transfer learning con MobileNetV2

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
aug = keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.08), layers.RandomZoom(0.1)])
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
entrada = keras.Input(shape=(IMG, IMG, 3))
x = aug(entrada)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(1, activation='sigmoid')(x)
modelo = keras.Model(entrada, salida)
modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
               metrics=['accuracy', keras.metrics.AUC(name='auc')])
modelo.summary()

## Celda 8 - Entrenar

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
pesos = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_df['caries'].values)
class_weight = {0: float(pesos[0]), 1: float(pesos[1])}
print('Pesos de clase:', class_weight)
parar = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
h1 = modelo.fit(ds_train, validation_data=ds_val, epochs=6, class_weight=class_weight, callbacks=[parar])

### Ajuste fino (descongelar las últimas capas)

In [ ]:
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(optimizer=keras.optimizers.Adam(1e-5), loss='binary_crossentropy',
               metrics=['accuracy', keras.metrics.AUC(name='auc')])
h2 = modelo.fit(ds_train, validation_data=ds_val, epochs=4, class_weight=class_weight, callbacks=[parar])

## Celda 9 - Evaluar con las imágenes de prueba

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
y_real = np.concatenate([y.numpy() for _, y in ds_test]).astype(int)
y_prob = modelo.predict(ds_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_real, y_pred, target_names=['sin caries', 'con caries'], digits=3))

cm = confusion_matrix(y_real, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['sin caries', 'con caries']).plot(cmap='Blues')
plt.title('Matriz de confusion (conjunto de prueba)')
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight'); plt.show()

hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['accuracy'], label='entrenamiento'); ax[0].plot(hist['val_accuracy'], label='validacion')
ax[0].set_title('Exactitud'); ax[0].set_xlabel('epoca'); ax[0].legend()
ax[1].plot(hist['loss'], label='entrenamiento'); ax[1].plot(hist['val_loss'], label='validacion')
ax[1].set_title('Perdida'); ax[1].set_xlabel('epoca'); ax[1].legend()
plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight'); plt.show()

## Celda 10 - Exportar el modelo (operación 15)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
metricas = {
    'imagenes_prueba': int(len(y_real)),
    'exactitud': round(float(accuracy_score(y_real, y_pred)), 4),
    'precision': round(float(precision_score(y_real, y_pred)), 4),
    'sensibilidad': round(float(recall_score(y_real, y_pred)), 4),
    'f1': round(float(f1_score(y_real, y_pred)), 4),
}
json.dump(metricas, open('metricas.json', 'w'), indent=2)
print(metricas)

modelo.save('caries_model.keras')
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('caries_model.tflite', 'wb').write(conv.convert())
!ls -lh caries_model.keras caries_model.tflite metricas.json

## Celda 11 - Probar una imagen y descargar los resultados

In [ ]:
from PIL import Image
muestra = test_df.sample(1, random_state=1).iloc[0]
img = np.array(Image.open(muestra['ruta']).convert('RGB').resize((IMG, IMG)))[None, ...].astype('float32')
p = float(modelo.predict(img, verbose=0)[0][0])
print('Imagen:', os.path.basename(muestra['ruta']))
print('Real  :', 'con caries' if muestra['caries'] else 'sin caries')
print(f'Modelo: {"con caries" if p >= 0.5 else "sin caries"} (probabilidad {p:.2%})')
plt.imshow(Image.open(muestra['ruta'])); plt.axis('off'); plt.show()

!zip -q resultados_modelo.zip caries_model.keras caries_model.tflite metricas.json matriz_confusion.png curvas_entrenamiento.png
from google.colab import files
files.download('resultados_modelo.zip')